# Lab 1: Find and download satellite imagery for the refinery sites

**Course:** Politics of World Energy Rivalry (Stanford, Policy)

**What you will do**

1. Load the six site boundaries (areas of interest, or AOIs) from the `AOI/` folder.
2. Search Planet's catalog for PlanetScope scenes over each AOI.
3. Pick the scenes best suited to measuring floating-roof oil tanks.
4. Order clipped copies of those scenes and download them.

**What you need**

- A Planet account with an API key (Stanford students: ask the Stanford Geospatial Center).
- Python 3.10 or newer with the packages in `notebooks/requirements.txt`.
- Set your key as an environment variable named `PL_API_KEY`, or paste it when the notebook asks.

**Why this matters for policy:** Tank fill levels are a public, independent signal of how much crude and product a refinery holds. Analysts use them to check company and government claims about output and supply.

## Concepts you need first

**AOI (area of interest).** A polygon that marks where you want data. Here each AOI is a 5 km by 5 km square centered on a refinery. The files are GeoJSON, a text format for shapes on a map.

**Scene.** One image taken by a satellite at one moment. Planet's PlanetScope satellites photograph the whole land surface about once a day at 3 m per pixel.

**Catalog search vs. order.** Searching only returns a list of scenes and their details (metadata). It is free and fast. Ordering is what creates files you can download, and it counts against your area quota.

**Metadata that matters for tanks**

| Field | Meaning | Why it matters |
|---|---|---|
| `cloud_cover` | Share of the scene covered by cloud, 0 to 1 | Cloud hides the tanks. |
| `sun_elevation` | Sun's angle above the horizon, in degrees | Low sun makes long shadows. Long shadows make fill levels easier to measure. |
| `sun_azimuth` | Compass direction of the sun, degrees clockwise from north | Tells you which way shadows fall. |
| `view_angle` | How far the satellite pointed off straight down | Large angles distort tank shapes. |
| `clear_percent` | Share of pixels with no cloud, haze or shadow | A second quality check. |

**API (application programming interface).** A way for code to talk to a web service. You send a request, you get data back. Your **API key** is a password for that service. Never paste it into a notebook you share.

**Quota.** Planet counts the area you download. A clipped 5 km by 5 km AOI is 25 km² per scene. Plan your orders before you submit them.

In [ ]:
# --- Imports ---------------------------------------------------------------
# Each import brings in a tool we use below.
import os            # read environment variables (where we keep the API key)
import json          # save and load small data files
import time          # pause between checks on an order
import getpass       # ask for a password without showing it on screen
from pathlib import Path   # work with file paths on any operating system

import requests      # send requests to web services
import pandas as pd  # tables of data (DataFrames)
import matplotlib.pyplot as plt   # charts
import geopandas as gpd           # tables of map data
from shapely.geometry import shape   # turn GeoJSON shapes into Python objects

## Step 1: Settings

Change these values to change your search. Everything else in the notebook reads from here.

In [ ]:
# The notebook lives in notebooks/, so the repo root is one folder up.
REPO = Path("..").resolve()
AOI_DIR = REPO / "AOI"                       # where the AOI files are
OUT_DIR = REPO / "data" / "imagery"          # where we save catalogs and images
OUT_DIR.mkdir(parents=True, exist_ok=True)   # create the folder if it is missing

# Search window. Winter months give lower sun and longer shadows in the north.
START_DATE = "2025-10-01"
END_DATE = "2026-10-01"

# Which Planet product to search.
# "PSScene" is PlanetScope (3 m). For SkySat (0.5 m) use "SkySatCollect".
# SkySat needs special access. Stanford students: ask the Geospatial Center.
ITEM_TYPE = "PSScene"

# Product bundle to order. This one is 8-band surface reflectance plus a cloud mask.
# For SkySat, check Planet's documentation for the matching bundle name.
PRODUCT_BUNDLE = "analytic_8b_sr_udm2"

# Search filters.
MAX_CLOUD = 0.2          # keep scenes with 20 percent cloud or less
MIN_COVERAGE = 0.95      # scene footprint must cover 95 percent of the AOI

# Rules for picking the best scenes for tank shadows.
BEST_MAX_CLOUD = 0.05    # tighter cloud limit for the final picks
BEST_MAX_VIEW_ANGLE = 10 # degrees off straight down
MIN_SUN_ELEV = 15        # below this, tall tanks shade their neighbors
MAX_SUN_ELEV = 45        # above this, shadows get too short to measure
SCENES_PER_SITE = 3      # how many scenes to order per site

# Safety switch. While True, the notebook prints the order and does NOT submit it.
DRY_RUN = True

## Step 2: Load the AOIs

Each `*_aoi.geojson` file holds one square. We read them into a Python dictionary keyed by a short site name (the "slug").

In [ ]:
aois = {}   # will map site slug -> information about that site

for path in sorted(AOI_DIR.glob("*_aoi.geojson")):
    data = json.loads(path.read_text())          # read the file as JSON
    feature = data["features"][0]                # each file has one feature
    slug = path.stem.replace("_aoi", "")         # e.g. "shell_pernis_netherlands"

    # Measure the area in km2. Degrees are not a unit of area, so we convert
    # the shape to a metric projection (UTM) first.
    gdf = gpd.GeoDataFrame(geometry=[shape(feature["geometry"])], crs="EPSG:4326")
    area_km2 = gdf.to_crs(gdf.estimate_utm_crs()).area.iloc[0] / 1e6

    aois[slug] = {
        "name": feature["properties"]["facility"],
        "lat": feature["properties"]["center_lat"],
        "lon": feature["properties"]["center_lon"],
        "geometry": feature["geometry"],          # GeoJSON dict, used by the API
        "shape": shape(feature["geometry"]),      # shapely object, used for math
        "area_km2": round(area_km2, 1),
    }

pd.DataFrame(aois).T[["name", "lat", "lon", "area_km2"]]

## Step 3: Set up the connection to Planet

`requests.Session` remembers the login for every call. Planet uses your API key as the user name and leaves the password blank.

In [ ]:
# Use the environment variable if it exists. Otherwise ask for the key.
PL_API_KEY = os.environ.get("PL_API_KEY") or getpass.getpass("Planet API key: ")

session = requests.Session()
session.auth = (PL_API_KEY, "")    # key as user name, empty password

DATA_URL = "https://api.planet.com/data/v1"                 # search service
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2" # ordering service

# Quick test: a good key returns status 200.
print("Connection status:", session.get(DATA_URL).status_code)

## Step 4: Search the catalog

A search request is a set of **filters** joined by AND. A scene must pass every filter to be returned. The filters here are:

- **GeometryFilter:** the scene must overlap the AOI.
- **DateRangeFilter:** the scene must be taken between the start and end dates.
- **RangeFilter on cloud cover:** the scene must have little cloud.
- **PermissionFilter:** your account must be allowed to download it.

Results come back in pages. The function follows the "next" link until there are no more pages.

In [ ]:
def build_search_request(geometry, start, end, max_cloud, item_type):
    """Build the JSON body for a Planet quick-search call."""
    geometry_filter = {"type": "GeometryFilter", "field_name": "geometry",
                       "config": geometry}
    date_filter = {"type": "DateRangeFilter", "field_name": "acquired",
                   "config": {"gte": f"{start}T00:00:00Z", "lte": f"{end}T23:59:59Z"}}
    cloud_filter = {"type": "RangeFilter", "field_name": "cloud_cover",
                    "config": {"lte": max_cloud}}
    permission_filter = {"type": "PermissionFilter", "config": ["assets:download"]}
    return {
        "item_types": [item_type],
        "filter": {"type": "AndFilter",
                   "config": [geometry_filter, date_filter, cloud_filter, permission_filter]},
    }


def run_search(request_body, max_pages=50):
    """Send the search and collect every page of results."""
    features = []
    resp = session.post(f"{DATA_URL}/quick-search", json=request_body)
    resp.raise_for_status()          # stop with a clear error if the call failed
    page = resp.json()
    for _ in range(max_pages):
        features.extend(page["features"])            # add this page's scenes
        next_url = page["_links"].get("_next")       # link to the next page
        if not next_url:                             # no link means we are done
            break
        resp = session.get(next_url)
        resp.raise_for_status()
        page = resp.json()
    return features


def features_to_table(features, aoi_shape):
    """Turn raw search results into a table with the fields we care about."""
    rows = []
    for f in features:
        p = f["properties"]                          # the scene's metadata
        footprint = shape(f["geometry"])             # the scene's outline
        # Share of the AOI that the scene covers. (Both shapes are in degrees,
        # which is fine for a ratio over an area this small.)
        coverage = footprint.intersection(aoi_shape).area / aoi_shape.area
        rows.append({
            "id": f["id"],
            "acquired": p["acquired"],
            "cloud_cover": p.get("cloud_cover"),
            "clear_percent": p.get("clear_percent"),
            "sun_elevation": p.get("sun_elevation"),
            "sun_azimuth": p.get("sun_azimuth"),
            "view_angle": p.get("view_angle"),
            "pixel_resolution": p.get("pixel_resolution"),
            "instrument": p.get("instrument"),
            "coverage": round(coverage, 3),
        })
    table = pd.DataFrame(rows)
    if not table.empty:
        table["acquired"] = pd.to_datetime(table["acquired"])   # text -> dates
        table = table.sort_values("acquired").reset_index(drop=True)
    return table

In [ ]:
catalogs = {}   # slug -> table of scenes

for slug, aoi in aois.items():
    body = build_search_request(aoi["geometry"], START_DATE, END_DATE, MAX_CLOUD, ITEM_TYPE)
    features = run_search(body)
    table = features_to_table(features, aoi["shape"])
    catalogs[slug] = table
    table.to_csv(OUT_DIR / f"catalog_{slug}.csv", index=False)   # save for notebook 3
    print(f"{aoi['name']:<35} {len(table):>4} scenes")

## Step 5: Look at what came back

The chart shows sun elevation for every scene. Each dot is one scene. Notice two things: how the sun angle changes through the year, and how it differs by latitude. Shell Pernis (52°N) sits much lower than Reliance Jamnagar (22°N).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, (slug, table) in zip(axes.ravel(), catalogs.items()):
    if table.empty:
        ax.set_title(f"{aois[slug]['name']} (no scenes)")
        continue
    ax.scatter(table["acquired"], table["sun_elevation"], s=8, alpha=0.6)
    ax.axhspan(MIN_SUN_ELEV, MAX_SUN_ELEV, color="green", alpha=0.1)  # target band
    ax.set_title(aois[slug]["name"], fontsize=10)
    ax.tick_params(axis="x", rotation=45)
axes[0, 0].set_ylabel("Sun elevation (degrees)")
axes[1, 0].set_ylabel("Sun elevation (degrees)")
fig.suptitle("Sun elevation of PlanetScope scenes (green band = target range)")
plt.tight_layout()
plt.show()

In [ ]:
# One summary row per site.
summary = []
for slug, table in catalogs.items():
    if table.empty:
        continue
    in_band = table["sun_elevation"].between(MIN_SUN_ELEV, MAX_SUN_ELEV)
    summary.append({
        "site": aois[slug]["name"],
        "scenes": len(table),
        "full_coverage": int((table["coverage"] >= MIN_COVERAGE).sum()),
        "median_sun_elev": round(table["sun_elevation"].median(), 1),
        "in_target_sun_band": int(in_band.sum()),
    })
pd.DataFrame(summary)

## Step 6: Pick the best scenes

For tank shadows we want scenes that:

- cover the whole AOI,
- have almost no cloud,
- look nearly straight down,
- have the sun between 15° and 45° (long enough shadows to measure, not so long that tanks shade each other).

Among scenes that pass, we take the ones with the **lowest sun**, because they give the longest shadows. The 15° and 45° limits are working assumptions. You will test them in Lab 3.

In [ ]:
def pick_best(table):
    """Return the best scenes for tank-shadow work from one site's table."""
    keep = table[
        (table["coverage"] >= MIN_COVERAGE)
        & (table["cloud_cover"] <= BEST_MAX_CLOUD)
        & (table["view_angle"] <= BEST_MAX_VIEW_ANGLE)
        & (table["sun_elevation"].between(MIN_SUN_ELEV, MAX_SUN_ELEV))
    ]
    # Lowest sun first, then take the top few.
    return keep.sort_values("sun_elevation").head(SCENES_PER_SITE)


selected = {}   # slug -> list of scene ids
for slug, table in catalogs.items():
    if table.empty:
        selected[slug] = []
        continue
    best = pick_best(table)
    selected[slug] = best["id"].tolist()
    print(f"\n{aois[slug]['name']}")
    print(best[["id", "acquired", "cloud_cover", "sun_elevation", "view_angle"]]
          .to_string(index=False) if len(best) else "  no scenes pass the filters")

# Save the picks so Lab 3 knows which scenes to expect.
(OUT_DIR / "selected_scenes.json").write_text(json.dumps(selected, indent=2))

**If a site has no scenes:** loosen one filter at a time (cloud, sun band, dates) and rerun from Step 4. Write down which filter you relaxed. That choice affects your results.

## Step 7: Order clipped imagery

An **order** asks Planet to prepare files. The **clip** tool cuts each scene to the AOI so you download 25 km² instead of a whole 600 km² scene. Orders run on Planet's servers. We submit, check every few seconds until it finishes, then download.

With `DRY_RUN = True` the code only prints what it would do and how much area it would use.

In [ ]:
def build_order(slug, scene_ids, geometry):
    """Build the JSON body for an order that clips scenes to the AOI."""
    return {
        "name": f"tank_lab_{slug}",
        "products": [{
            "item_ids": scene_ids,
            "item_type": ITEM_TYPE,
            "product_bundle": PRODUCT_BUNDLE,
        }],
        "tools": [{"clip": {"aoi": geometry}}],
    }


def submit_order(order):
    """Send the order to Planet and return its id."""
    resp = session.post(ORDERS_URL, json=order)
    resp.raise_for_status()
    return resp.json()["id"]


def wait_for_order(order_id, poll_seconds=15, timeout_minutes=30):
    """Check the order until it finishes or time runs out."""
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        info = session.get(f"{ORDERS_URL}/{order_id}").json()
        print("  order state:", info["state"])
        if info["state"] in ("success", "partial", "failed", "cancelled"):
            return info
        time.sleep(poll_seconds)
    raise TimeoutError("Order did not finish in time. Check it later with its id.")


def download_order(info, dest_folder):
    """Download every file in a finished order."""
    dest_folder.mkdir(parents=True, exist_ok=True)
    for item in info["_links"].get("results", []):
        target = dest_folder / Path(item["name"]).name   # keep just the file name
        if target.exists():                              # skip files we already have
            continue
        # The download link is pre-signed, so we use plain requests (no login).
        with requests.get(item["location"], stream=True) as r:
            r.raise_for_status()
            with open(target, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
        print("  saved", target.name)

In [ ]:
total_area = 0
for slug, ids in selected.items():
    if not ids:
        continue
    area = len(ids) * aois[slug]["area_km2"]     # clipped area counts toward quota
    total_area += area
    order = build_order(slug, ids, aois[slug]["geometry"])
    print(f"{aois[slug]['name']}: {len(ids)} scenes, {area:.0f} km2")

    if DRY_RUN:
        continue                                   # do nothing more in dry-run mode

    order_id = submit_order(order)
    print("  submitted order", order_id)
    info = wait_for_order(order_id)
    if info["state"] in ("success", "partial"):
        download_order(info, OUT_DIR / slug)
    else:
        print("  order did not succeed:", info["state"])

print(f"\nTotal area for this run: {total_area:.0f} km2")
if DRY_RUN:
    print("DRY_RUN is True. Set it to False in Step 1 to submit the orders.")

## Check your work

After a real run, each site folder in `data/imagery/` should hold `.tif` image files (the clipped scenes) plus small `.json` and `.xml` files with metadata. Open one `.tif` in QGIS to confirm the refinery is in the frame.

**Questions for discussion**

1. Which site has the most usable scenes? Which has the fewest? What explains the difference (cloud, latitude, orbit)?
2. Why does the lab pick the lowest sun angle? What is the cost of going too low?
3. Planet's license limits sharing of imagery. How does that shape what a policy analyst can publish from this work?
4. If the Jingbo site has few usable scenes, what other evidence could you use to check its output?

**Next:** `02_tank_inventory_osm.ipynb` finds where the tanks are. For a free alternative to Planet imagery, see `04_sentinel2_earth_engine_tank_metrics.ipynb`.